In [ ]:
### Causal Inference Models

## Difference-in-Difference (DiD) Analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Filter selected contract types
df_did = df_encoded[df_encoded['Contract'].isin([0, 1])].copy()

# Create tenure groups using median split
median_tenure = df_did['tenure'].median()
df_did['tenure_group'] = np.where(
    df_did['tenure'] <= median_tenure,
    'before',
    'after'
)

# Compute average churn rates
churn_rates = (
    df_did
    .groupby(['Contract', 'tenure_group'])['Churn']
    .mean()
    .unstack()
)

# Difference-in-Difference estimator
did = (
    (churn_rates.loc[1, 'after'] - churn_rates.loc[1, 'before']) -
    (churn_rates.loc[0, 'after'] - churn_rates.loc[0, 'before'])
)

print(f'DiD estimator: {did:.4f}')

# Visualisation
plt.figure(figsize=(8, 5))

plt.plot(
    ['before', 'after'],
    churn_rates.loc[0],
    marker='o',
    linestyle='--',
    label='Control: Month-to-month'
)

plt.plot(
    ['before', 'after'],
    churn_rates.loc[1],
    marker='o',
    linestyle='-',
    label='Treatment: One year'
)

plt.title('Difference-in-Difference Analysis of Customer Churn')
plt.xlabel('Tenure Group')
plt.ylabel('Average Churn Rate')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()




## Regression Discontinuity Design (RDD)

cutoff = 61

df_encoded['monthly_charges_centered'] = (
    df_encoded['MonthlyCharges'] - cutoff
)

df_encoded['treatment'] = (
    df_encoded['MonthlyCharges'] >= cutoff
).astype(int)

# Bin observations around cutoff
bins = pd.cut(df_encoded['monthly_charges_centered'], bins=50)

binned = (
    df_encoded
    .groupby(bins)['Churn']
    .mean()
    .reset_index()
)

# Visualisation
plt.figure(figsize=(10, 6))

plt.scatter(
    range(len(binned)),
    binned['Churn'],
    alpha=0.7,
    label='Binned churn rate'
)

plt.axvline(
    x=len(binned) // 2,
    linestyle=':',
    label='Cutoff'
)

plt.title('RDD Analysis of Churn Around Monthly Charge Cutoff')
plt.ylabel('Average Churn Rate')
plt.legend()
plt.tight_layout()
plt.show()




## Two-Stage Least Squares (2SLS / IV)

import statsmodels.formula.api as smf

endogenous = 'OnlineSecurity'
outcome = 'Churn'
instrument = 'Partner'

controls = [
    col for col in df_encoded.columns
    if col not in [endogenous, outcome, instrument]
]

# First stage
first_stage = smf.ols(
    f"{endogenous} ~ {instrument} + {' + '.join(controls)}",
    data=df_encoded
).fit()

df_encoded['security_predicted'] = first_stage.predict(df_encoded)

# Second stage
second_stage = smf.ols(
    f"{outcome} ~ security_predicted + {' + '.join(controls)}",
    data=df_encoded
).fit()

print(second_stage.summary())
